# TAREA 5.5: Sistema experto basado en probabilidad para diagnóstico industrial
**Módulo:** Modelos de Inteligencia Artificial  
**Curso:** Especialización en IA y Big Data  

---

## 1. Introducción
Sistema Experto Basado en Conocimiento (SBC) con enfoque probabilístico usando **redes Bayesianas** (`pgmpy`) para mantenimiento predictivo industrial (Industria 4.0).

A diferencia de la versión básica (probabilidades condicionales simples), esta implementación permite razonar con **múltiples síntomas simultáneos** aplicando correctamente el Teorema de Bayes mediante eliminación de variables.

El sistema identifica qué pieza falla a partir de:
1. **Síntomas** observados por el operario (uno o varios a la vez).
2. **Red Bayesiana** con CPTs que codifican la relación pieza→síntoma.
3. **Desgaste real** de los componentes (horas de uso vs. horas límite).

Incorpora **aprendizaje por retroalimentación**: cada diagnóstico confirmado ajusta las CPTs y actualiza el estado físico de las piezas.

In [5]:
!pip install pgmpy pandas -q

In [11]:
import json
import os
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [12]:
class SistemaExpertoBayesiano:
    FACTOR_DESGASTE = 1.5
    INCREMENTO_APRENDIZAJE = 0.05
    PROB_MAX = 0.95
    PROB_MIN = 0.01
    PIEZAS = ["Motor", "Correa", "Filtro"]
    SINTOMAS = ["Vibracion", "Ruido_Agudo", "Sobrecalentamiento"]

    def __init__(self, archivo_db='base_conocimiento_bayesiana.json'):
        self.archivo_db = archivo_db
        self.datos = self.cargar_datos()
        self.modelo = None
        self.inferencia = None
        self._construir_red()

    def cargar_datos(self):
        if os.path.exists(self.archivo_db):
            with open(self.archivo_db, 'r', encoding='utf-8') as f:
                return json.load(f)

        return {
            "piezas": {
                "Motor":  {"horas_limite": 5000, "horas_actuales": 4800, "fallos_acumulados": 0},
                "Correa": {"horas_limite": 1000, "horas_actuales": 950,  "fallos_acumulados": 0},
                "Filtro": {"horas_limite": 2000, "horas_actuales": 100,  "fallos_acumulados": 0}
            },
            "probabilidades": {
                "Vibracion":          {"Motor": 0.6, "Correa": 0.3, "Filtro": 0.1},
                "Ruido_Agudo":        {"Motor": 0.2, "Correa": 0.7, "Filtro": 0.1},
                "Sobrecalentamiento": {"Motor": 0.7, "Correa": 0.1, "Filtro": 0.2}
            }
        }

    def guardar_datos(self):
        with open(self.archivo_db, 'w', encoding='utf-8') as f:
            json.dump(self.datos, f, indent=4, ensure_ascii=False)

    def _construir_red(self):
        """Construye la red bayesiana Pieza -> {Síntomas} a partir de la base de conocimiento."""
        self.modelo = DiscreteBayesianNetwork([("Pieza", s) for s in self.SINTOMAS])

        cpd_pieza = TabularCPD(
            variable="Pieza",
            variable_card=3,
            values=[[1/3], [1/3], [1/3]],
            state_names={"Pieza": self.PIEZAS}
        )
        self.modelo.add_cpds(cpd_pieza)

        for sintoma in self.SINTOMAS:
            probs_si = [self.datos["probabilidades"][sintoma][p] for p in self.PIEZAS]

            for i, pieza in enumerate(self.PIEZAS):
                info = self.datos["piezas"][pieza]
                if info["horas_actuales"] > info["horas_limite"]:
                    probs_si[i] = min(0.99, probs_si[i] * self.FACTOR_DESGASTE)

            probs_no = [1.0 - p for p in probs_si]

            cpd = TabularCPD(
                variable=sintoma,
                variable_card=2,
                values=[probs_no, probs_si],
                evidence=["Pieza"],
                evidence_card=[3],
                state_names={sintoma: ["No", "Si"], "Pieza": self.PIEZAS}
            )
            self.modelo.add_cpds(cpd)

        assert self.modelo.check_model()
        self.inferencia = VariableElimination(self.modelo)

    def diagnosticar(self, sintomas_observados):
        invalidos = [s for s in sintomas_observados if s not in self.SINTOMAS]
        if invalidos:
            return None, invalidos

        evidencia = {s: "Si" for s in sintomas_observados}
        resultado = self.inferencia.query(variables=["Pieza"], evidence=evidencia)

        resultados = []
        for i, pieza in enumerate(self.PIEZAS):
            info = self.datos["piezas"][pieza]
            prob = resultado.values[i]
            resultados.append({
                "Pieza": pieza,
                "P(Pieza|Síntomas)": round(prob, 4),
                "Desgaste": f"{info['horas_actuales']}/{info['horas_limite']}",
                "Fallos_Previos": info["fallos_acumulados"]
            })

        return sorted(resultados, key=lambda x: x["P(Pieza|Síntomas)"], reverse=True), []

    def retroalimentacion(self, sintomas_observados, pieza_real):
        if pieza_real not in self.datos["piezas"]:
            raise ValueError(f"Pieza '{pieza_real}' no registrada.")

        for sintoma in sintomas_observados:
            if sintoma not in self.datos["probabilidades"]:
                raise ValueError(f"Síntoma '{sintoma}' no registrado.")

            probs = self.datos["probabilidades"][sintoma]
            nuevo_valor = min(self.PROB_MAX, probs[pieza_real] + self.INCREMENTO_APRENDIZAJE)
            delta = nuevo_valor - probs[pieza_real]
            probs[pieza_real] = nuevo_valor

            otras = {p: v for p, v in probs.items() if p != pieza_real}
            suma_otras = sum(otras.values())
            if suma_otras > 0 and delta > 0:
                for p in otras:
                    reduccion = delta * (otras[p] / suma_otras)
                    probs[p] = max(self.PROB_MIN, probs[p] - reduccion)

        self.datos["piezas"][pieza_real]["horas_actuales"] = 0
        self.datos["piezas"][pieza_real]["fallos_acumulados"] += 1

        self.guardar_datos()
        self._construir_red()
        print(f"\n[SISTEMA] Aprendizaje completado. Red Bayesiana reconstruida.")

In [13]:
# Ahora realizamos una demostración completa del sistema experto bayesiano, cubriendo persistencia, inferencia y aprendizaje.
if os.path.exists('base_conocimiento_bayesiana.json'):
    os.remove('base_conocimiento_bayesiana.json')

sistema = SistemaExpertoBayesiano()

print("="*60)
print("1) PERSISTENCIA: No existe JSON -> se crean datos por defecto")
print("="*60)
print(f"Archivo existe tras init: {os.path.exists('base_conocimiento_bayesiana.json')}")
sistema.guardar_datos()
print(f"Archivo existe tras guardar: {os.path.exists('base_conocimiento_bayesiana.json')}")

1) PERSISTENCIA: No existe JSON -> se crean datos por defecto
Archivo existe tras init: False
Archivo existe tras guardar: True


In [14]:
print("="*60)
print("2) INFERENCIA: Un solo síntoma -> 'Vibracion'")
print("="*60)
resultado, _ = sistema.diagnosticar(["Vibracion"])
print(pd.DataFrame(resultado).to_string(index=False))

2) INFERENCIA: Un solo síntoma -> 'Vibracion'
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor                0.6 4800/5000               0
Correa                0.3  950/1000               0
Filtro                0.1  100/2000               0


In [15]:
print("="*60)
print("3) INFERENCIA: Múltiples síntomas -> 'Vibracion' + 'Sobrecalentamiento'")
print("="*60)
resultado2, _ = sistema.diagnosticar(["Vibracion", "Sobrecalentamiento"])
print(pd.DataFrame(resultado2).to_string(index=False))

3) INFERENCIA: Múltiples síntomas -> 'Vibracion' + 'Sobrecalentamiento'
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor             0.8936 4800/5000               0
Correa             0.0638  950/1000               0
Filtro             0.0426  100/2000               0


In [16]:
print("="*60)
print("4) INFERENCIA: Todos los síntomas simultáneos")
print("="*60)
resultado3, _ = sistema.diagnosticar(["Vibracion", "Ruido_Agudo", "Sobrecalentamiento"])
print(pd.DataFrame(resultado3).to_string(index=False))

4) INFERENCIA: Todos los síntomas simultáneos
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor             0.7850 4800/5000               0
Correa             0.1963  950/1000               0
Filtro             0.0187  100/2000               0


In [17]:
print("="*60)
print("5) APRENDIZAJE: El técnico confirma que fue la Correa (síntoma: Vibracion)")
print("="*60)
print(f"ANTES -> P(Correa|Vibracion) en CPT = {sistema.datos['probabilidades']['Vibracion']['Correa']}")
print(f"ANTES -> Horas Correa = {sistema.datos['piezas']['Correa']['horas_actuales']}")
sistema.retroalimentacion(["Vibracion"], "Correa")
print(f"DESPUÉS -> P(Correa|Vibracion) en CPT = {sistema.datos['probabilidades']['Vibracion']['Correa']}")
print(f"DESPUÉS -> Horas Correa = {sistema.datos['piezas']['Correa']['horas_actuales']}")

print("\nDiagnóstico post-aprendizaje para 'Vibracion':")
resultado_post, _ = sistema.diagnosticar(["Vibracion"])
print(pd.DataFrame(resultado_post).to_string(index=False))

5) APRENDIZAJE: El técnico confirma que fue la Correa (síntoma: Vibracion)
ANTES -> P(Correa|Vibracion) en CPT = 0.3
ANTES -> Horas Correa = 950

[SISTEMA] Aprendizaje completado. Red Bayesiana reconstruida.
DESPUÉS -> P(Correa|Vibracion) en CPT = 0.35
DESPUÉS -> Horas Correa = 0

Diagnóstico post-aprendizaje para 'Vibracion':
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor             0.5571 4800/5000               0
Correa             0.3500    0/1000               1
Filtro             0.0929  100/2000               0


In [18]:
print("="*60)
print("6) PERSISTENCIA: Recargamos desde JSON y verificamos cambios")
print("="*60)
sistema2 = SistemaExpertoBayesiano()
print(f"P(Correa|Vibracion) recargada = {sistema2.datos['probabilidades']['Vibracion']['Correa']}")
print(f"Horas Correa recargada = {sistema2.datos['piezas']['Correa']['horas_actuales']}")
print(f"Fallos acumulados Correa = {sistema2.datos['piezas']['Correa']['fallos_acumulados']}")
print("-> Los datos persisten correctamente en disco.")

6) PERSISTENCIA: Recargamos desde JSON y verificamos cambios
P(Correa|Vibracion) recargada = 0.35
Horas Correa recargada = 0
Fallos acumulados Correa = 1
-> Los datos persisten correctamente en disco.


In [19]:
print("ESTADO ACTUAL DE LA BASE DE CONOCIMIENTO")
df_piezas = pd.DataFrame(sistema.datos["piezas"]).T
df_piezas["% desgaste"] = (df_piezas["horas_actuales"] / df_piezas["horas_limite"] * 100).round(1)
display(df_piezas)

print("\nMATRIZ DE PROBABILIDADES (CPTs base)")
df_probs = pd.DataFrame(sistema.datos["probabilidades"])
df_probs.loc["TOTAL"] = df_probs.sum()
display(df_probs.round(3))

ESTADO ACTUAL DE LA BASE DE CONOCIMIENTO


,horas_limite,horas_actuales,fallos_acumulados,% desgaste
Motor,5000,4800,0,96.0
Correa,1000,0,1,0.0
Filtro,2000,100,0,5.0



MATRIZ DE PROBABILIDADES (CPTs base)


,Vibracion,Ruido_Agudo,Sobrecalentamiento
Motor,0.557,0.2,0.7
Correa,0.350,0.7,0.1
Filtro,0.093,0.1,0.2
TOTAL,1.000,1.0,1.0


In [20]:
sistema.datos["piezas"]["Motor"]["horas_actuales"] = 5200
sistema.guardar_datos()
sistema._construir_red()

print("Diagnóstico con Motor en sobreuso (5200/5000):")

print("\nSolo Sobrecalentamiento:")
r1, _ = sistema.diagnosticar(["Sobrecalentamiento"])
print(pd.DataFrame(r1).to_string(index=False))

print("\nSobrecalentamiento + Vibracion:")
r2, _ = sistema.diagnosticar(["Sobrecalentamiento", "Vibracion"])
print(pd.DataFrame(r2).to_string(index=False))

Diagnóstico con Motor en sobreuso (5200/5000):

Solo Sobrecalentamiento:
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor             0.7674 5200/5000               0
Filtro             0.1550  100/2000               0
Correa             0.0775    0/1000               1

Sobrecalentamiento + Vibracion:
 Pieza  P(Pieza|Síntomas)  Desgaste  Fallos_Previos
 Motor             0.9392 5200/5000               0
Correa             0.0397    0/1000               1
Filtro             0.0211  100/2000               0


## Conclusiones y ampliación

**Ventajas de la red bayesiana sobre el sistema básico:**
- Razonamiento con múltiples síntomas simultáneos aplicando Bayes correctamente.
- Las probabilidades posteriores reflejan la influencia conjunta de todas las evidencias.
- Ejemplo: Vibración + Sobrecalentamiento juntos apuntan al Motor con más fuerza que cualquiera por separado.

**Limitaciones que persisten:**
- Las CPTs se ajustan con incrementos fijos (+5%), no con estimación estadística formal.
- El prior sobre las piezas es uniforme; en un entorno real se estimaría a partir del histórico de fallos.

**Posibles mejoras:**
- Estimación de CPTs a partir de datos históricos con `BayesianEstimator` de pgmpy.
- Incorporar variables continuas (temperatura, Hz) mediante discretización.
- Añadir nodos intermedios (estado de desgaste como variable discreta en la red).

## Recursos y Enlaces de Interés
* [Sistemas Expertos Probabilísticos y Redes Bayesianas](https://iturbide.org/sistemas-expertos-probabilisticos/)
* [Teorema de Bayes explicado de forma sencilla](https://recursos.citic.es/ia/bayes)
* [Documentación pgmpy](https://pgmpy.org/)
* [Documentación oficial de la librería JSON](https://docs.python.org/3/library/json.html)
* [Netica (para modelado visual de probabilidades)](https://www.norsys.com/netica.html)